# Notebook 6 — Confidence Calibration
**RSNA Intracranial Hemorrhage Detection**

A well-calibrated model outputs probabilities that match empirical frequencies.
An overconfident model may output p=0.95 for cases that are only 60% correct.

This notebook:
1. Measures raw calibration (ECE, reliability diagram)
2. Applies **Temperature Scaling** (parametric, fast)
3. Applies **Isotonic Regression** (non-parametric, flexible)
4. Compares all three and selects the best method
5. Defines and validates the **3-band triage confidence system**
6. Saves calibration parameters for use in the report generator

### Calibration circularity caveat
> **Note:** Temperature scaling and isotonic regression are both fitted and evaluated
> on the same validation set. This means reported post-calibration ECE is slightly
> optimistic. In a production setting, a held-out calibration set should be used.
> However, with 100K+ validation samples, the optimism is minimal.

### Required inputs
- NB02 output: `manifest.csv` + `cache/` NPY arrays + `normalization_stats.json`
- NB03 output: `best_model.pth`, `checkpoint.pth`

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────
import os, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, brier_score_loss
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm.auto import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

CACHE_INPUT_DIR = '/kaggle/input/notebooks/harshitghosh/nb02eda'
NPY_CACHE_DIR   = f'{CACHE_INPUT_DIR}/cache'
MANIFEST_PATH   = f'{CACHE_INPUT_DIR}/manifest.csv'
MODEL_PATH      = '/kaggle/input/ich-train-session-final/best_model.pth'
CHECKPOINT      = '/kaggle/input/ich-train-session-final/checkpoint.pth'

ARCH        = 'efficientnet_b0'
IMG_SIZE    = 256
BATCH_SIZE  = 64
NUM_WORKERS = 4
SEED        = 42
ECE_BINS    = 15    # number of equal-width bins for ECE

# ─── Load normalization stats ────────────────────────────────────────────
_norm_path = os.path.join(CACHE_INPUT_DIR, 'normalization_stats.json')
if os.path.exists(_norm_path):
    with open(_norm_path) as f:
        _norm = json.load(f)
    MEAN = _norm['mean']
    STD  = _norm['std']
    print(f'Dataset normalization: mean={MEAN}, std={STD}')
else:
    MEAN = [0.485, 0.456, 0.406]
    STD  = [0.229, 0.224, 0.225]
    print(f'Using ImageNet defaults: mean={MEAN}, std={STD}')

In [ ]:
# ── 1. Load model & get raw logits on validation set ─────────────────────
def build_model(arch):
    if arch == 'efficientnet_b0':
        m = models.efficientnet_b0(weights=None)
        m.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.classifier[1].in_features, 1))
    elif arch == 'resnet50':
        m = models.resnet50(weights=None)
        m.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.fc.in_features, 1))
    return m


class ICHDataset(Dataset):
    def __init__(self, df, npy_root, transform):
        self.df = df.reset_index(drop=True)
        self.npy_root = npy_root
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        path = os.path.join(self.npy_root, f'{row["image_id"]}.npy')
        try:
            img = np.load(path)  # uint8  [0, 255]
        except Exception:
            img = np.zeros((IMG_SIZE, IMG_SIZE, 3), np.uint8)
        return self.transform(img), torch.tensor(float(row['any']), dtype=torch.float32)


model = build_model(ARCH)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model = model.to(DEVICE).eval()

ckpt = torch.load(CHECKPOINT, map_location='cpu')
BASE_THRESHOLD = ckpt.get('best_thresh', 0.5)

manifest = pd.read_csv(MANIFEST_PATH)
val_df   = manifest[manifest['split'] == 'val'].reset_index(drop=True)

val_transform = T.Compose([T.ToPILImage(), T.ToTensor(), T.Normalize(mean=MEAN, std=STD)])
val_ds = ICHDataset(val_df, NPY_CACHE_DIR, val_transform)
val_loader = DataLoader(val_ds, BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

@torch.no_grad()
def collect_logits(model, loader):
    logits_, labels_ = [], []
    for imgs, lbls in tqdm(loader, desc='Collecting logits'):
        with torch.amp.autocast(device_type='cuda'):
            out = model(imgs.to(DEVICE)).squeeze(1).cpu().float()
        logits_.append(out); labels_.append(lbls)
    return torch.cat(logits_).numpy(), torch.cat(labels_).numpy()


raw_logits, labels = collect_logits(model, val_loader)
raw_probs = torch.sigmoid(torch.tensor(raw_logits)).numpy()

print(f'Collected logits for {len(labels):,} samples')
print(f'Raw AUC: {roc_auc_score(labels, raw_probs):.5f}')

In [ ]:
# ── 2. ECE calculation ────────────────────────────────────────────────────
def expected_calibration_error(probs: np.ndarray,
                                labels: np.ndarray,
                                n_bins: int = 15) -> tuple:
    """
    Compute ECE and return (ece, bin_confs, bin_accs, bin_sizes) for plotting.
    """
    bins      = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids   = np.digitize(probs, bins, right=True) - 1
    bin_ids   = np.clip(bin_ids, 0, n_bins - 1)

    bin_confs = np.zeros(n_bins)
    bin_accs  = np.zeros(n_bins)
    bin_sizes = np.zeros(n_bins)

    for b in range(n_bins):
        mask = bin_ids == b
        if mask.sum() == 0:
            continue
        bin_sizes[b] = mask.sum()
        bin_confs[b] = probs[mask].mean()
        bin_accs[b]  = labels[mask].mean()

    ece = (np.abs(bin_accs - bin_confs) * bin_sizes / len(probs)).sum()
    return float(ece), bin_confs, bin_accs, bin_sizes


raw_ece, raw_bc, raw_ba, raw_bs = expected_calibration_error(raw_probs, labels)
print(f'Raw ECE: {raw_ece:.5f}  (lower is better; 0 = perfect calibration)')

In [ ]:
# ── 3. Temperature Scaling ────────────────────────────────────────────────
class TemperatureScaler(nn.Module):
    """Single temperature parameter T that scales all logits before sigmoid."""
    def __init__(self):
        super().__init__()
        self.temperature = nn.Parameter(torch.ones(1) * 1.5)

    def forward(self, logits):
        return logits / self.temperature


def fit_temperature_scaling(logits: np.ndarray, labels: np.ndarray,
                              lr: float = 0.01, n_iter: int = 200) -> float:
    ts_model  = TemperatureScaler()
    optimizer = torch.optim.LBFGS([ts_model.temperature], lr=lr, max_iter=n_iter)
    logits_t  = torch.tensor(logits, dtype=torch.float32)
    labels_t  = torch.tensor(labels, dtype=torch.float32)
    criterion = nn.BCEWithLogitsLoss()

    def closure():
        optimizer.zero_grad()
        loss = criterion(ts_model(logits_t), labels_t)
        loss.backward()
        return loss

    optimizer.step(closure)
    T = ts_model.temperature.item()
    return T


T_opt = fit_temperature_scaling(raw_logits, labels)
ts_logits = raw_logits / T_opt
ts_probs  = torch.sigmoid(torch.tensor(ts_logits)).numpy()

ts_ece, ts_bc, ts_ba, ts_bs = expected_calibration_error(ts_probs, labels)
print(f'Optimal temperature T : {T_opt:.4f}')
print(f'Temperature-scaled ECE: {ts_ece:.5f}')

In [ ]:
# ── 4. Isotonic Regression ────────────────────────────────────────────────
ir     = IsotonicRegression(out_of_bounds='clip')
ir.fit(raw_probs, labels)
ir_probs = ir.predict(raw_probs)

ir_ece, ir_bc, ir_ba, ir_bs = expected_calibration_error(ir_probs, labels)
print(f'Isotonic Regression ECE: {ir_ece:.5f}')

In [ ]:
# ── 5. Summary table & reliability diagrams ───────────────────────────────
summary = pd.DataFrame([
    {'Method': 'Raw (uncalibrated)',    'ECE': raw_ece,
     'Brier': brier_score_loss(labels, raw_probs),
     'AUC':   roc_auc_score(labels, raw_probs)},
    {'Method': 'Temperature Scaling',  'ECE': ts_ece,
     'Brier': brier_score_loss(labels, ts_probs),
     'AUC':   roc_auc_score(labels, ts_probs)},
    {'Method': 'Isotonic Regression',  'ECE': ir_ece,
     'Brier': brier_score_loss(labels, ir_probs),
     'AUC':   roc_auc_score(labels, ir_probs)},
])
for col in ['ECE', 'Brier', 'AUC']:
    summary[col] = summary[col].round(5)

print('Calibration comparison:')
print(summary.to_string(index=False))

# Reliability diagrams
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

def plot_reliability(ax, bin_confs, bin_accs, bin_sizes, ece, title):
    mask = bin_sizes > 0
    ax.bar(bin_confs[mask], bin_accs[mask],
           width=1.0/ECE_BINS, alpha=0.6, color='tab:blue',
           align='center', label='Accuracy per bin')
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Perfect calibration')
    gap_x = np.linspace(0, 1, 100)
    ax.fill_between(gap_x, gap_x, alpha=0.1, color='orange', label='Calibration gap')
    ax.set(title=f'{title}\nECE={ece:.4f}',
           xlabel='Mean predicted confidence', ylabel='Fraction of positives')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.legend(fontsize=8)

plot_reliability(*([axes[0]] + list(expected_calibration_error(raw_probs, labels))),
                 'Raw')
plot_reliability(*([axes[1]] + list(expected_calibration_error(ts_probs, labels))),
                 'Temperature Scaled')
plot_reliability(*([axes[2]] + list(expected_calibration_error(ir_probs, labels))),
                 'Isotonic Regression')

plt.suptitle('Reliability Diagrams', fontsize=13)
plt.tight_layout()
plt.savefig('/kaggle/working/reliability_diagrams.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 6. Choose best calibration method & define confidence bands ───────────
# Use Temperature Scaling as the deployed method:
#   - Simple (1 parameter), generalises well, preserves AUC
#   - Isotonic can overfit on smaller val sets
# Change to 'isotonic' if ir_ece is substantially better

CALIBRATION_METHOD = 'temperature'   # options: 'none', 'temperature', 'isotonic'

if CALIBRATION_METHOD == 'temperature':
    cal_probs = ts_probs
    best_ece  = ts_ece
elif CALIBRATION_METHOD == 'isotonic':
    cal_probs = ir_probs
    best_ece  = ir_ece
else:
    cal_probs = raw_probs
    best_ece  = raw_ece

# ── 3-band triage system ──────────────────────────────────────────────────
#  HIGH   : cal_prob >= 0.75  → haemorrhage detected, URGENT review
#  MEDIUM : 0.35 <= cal_prob < 0.75 → uncertain, standard review
#  LOW    : cal_prob < 0.35   → likely normal, routine workflow

HIGH_THRESHOLD   = 0.75
LOW_THRESHOLD    = 0.35

def assign_band(p):
    if p >= HIGH_THRESHOLD:
        return 'HIGH'
    elif p >= LOW_THRESHOLD:
        return 'MEDIUM'
    else:
        return 'LOW'

bands = np.vectorize(assign_band)(cal_probs)

band_df = pd.DataFrame({
    'label'   : labels,
    'cal_prob': cal_probs,
    'band'    : bands,
})

print('Band distribution:')
print(band_df['band'].value_counts().to_frame())
print(f'\n{CALIBRATION_METHOD.upper()} ECE: {best_ece:.5f}')

In [ ]:
# ── 7. Per-band error rates ───────────────────────────────────────────────
band_stats = []
for band_name in ['HIGH', 'MEDIUM', 'LOW']:
    sub = band_df[band_df['band'] == band_name]
    if len(sub) == 0: continue
    n_pos      = int(sub['label'].sum())
    n_neg      = int((sub['label'] == 0).sum())
    n_total    = len(sub)
    pos_rate   = n_pos / n_total

    # For HIGH band: FN rate (fraction of actual positives predicted in a lower band)
    # For LOW  band: FP rate (fraction of actual negatives predicted in a higher band)
    # Here we just report positive rate and fraction of cases in each band.
    band_stats.append({
        'Band'         : band_name,
        'N cases'      : n_total,
        '% of val set' : round(n_total / len(band_df) * 100, 1),
        'Positive rate': round(pos_rate, 4),
        'N positive'   : n_pos,
        'N negative'   : n_neg,
    })

band_summary = pd.DataFrame(band_stats)
print('\nPer-band statistics:')
print(band_summary.to_string(index=False))
print()
print('Interpretation:')
print(f'  HIGH band positive rate ≈ PPV for urgent review triage')
print(f'  LOW  band positive rate ≈ miss rate for de-prioritised cases')
print(f'  MEDIUM band = requires manual review — maximise routing here for safety')

In [ ]:
# ── 8. Visualise confidence band distribution ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

band_colors = {'HIGH': '#e74c3c', 'MEDIUM': '#f39c12', 'LOW': '#2ecc71'}

# Histogram of calibrated probs, coloured by band
for band_name, color in band_colors.items():
    sub = band_df[band_df['band'] == band_name]['cal_prob']
    axes[0].hist(sub, bins=40, alpha=0.7, color=color,
                 label=f'{band_name} (n={len(sub)})')
axes[0].axvline(HIGH_THRESHOLD, color='red',   linestyle='--', linewidth=1)
axes[0].axvline(LOW_THRESHOLD,  color='green', linestyle='--', linewidth=1)
axes[0].set(title='Calibrated probability distribution by band',
            xlabel='Calibrated probability', ylabel='Count')
axes[0].legend()

# Positive rate per band
axes[1].bar(band_summary['Band'], band_summary['Positive rate'],
            color=[band_colors[b] for b in band_summary['Band']])
for i, row in band_summary.iterrows():
    axes[1].text(i, row['Positive rate'] + 0.01,
                 f'{row["Positive rate"]*100:.1f}%',
                 ha='center', fontsize=10, fontweight='bold')
axes[1].set(title='Hemorrhage positive rate per confidence band',
            ylabel='Positive rate', ylim=(0, 1.1))

plt.suptitle(f'Confidence Band Analysis ({CALIBRATION_METHOD.title()} calibration)', fontsize=12)
plt.tight_layout()
plt.savefig('/kaggle/working/confidence_bands.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 9. Save calibration parameters ───────────────────────────────────────
import joblib

calibration_params = {
    'method'         : CALIBRATION_METHOD,
    'temperature'    : float(T_opt),
    'base_threshold' : float(BASE_THRESHOLD),
    'high_threshold' : float(HIGH_THRESHOLD),
    'low_threshold'  : float(LOW_THRESHOLD),
    'raw_ece'        : float(raw_ece),
    'cal_ece'        : float(best_ece),
}

with open('/kaggle/working/calibration_params.json', 'w') as f:
    json.dump(calibration_params, f, indent=2)

# Save isotonic regressor (useful as fallback)
joblib.dump(ir, '/kaggle/working/isotonic_regressor.pkl')

print('Saved:')
print('  /kaggle/working/calibration_params.json')
print('  /kaggle/working/isotonic_regressor.pkl')
print('  /kaggle/working/reliability_diagrams.png')
print('  /kaggle/working/confidence_bands.png')
print()
print(json.dumps(calibration_params, indent=2))

In [ ]:
# ── HEALTH CHECK — automated output validation ────────────────────────────

errors = []

# Check calibration params
cal_hc_path = '/kaggle/working/calibration_params.json'
if not os.path.exists(cal_hc_path):
    errors.append('calibration_params.json is MISSING')
else:
    with open(cal_hc_path) as f:
        cal_hc = json.load(f)
    if cal_hc.get('temperature', 0) <= 0:
        errors.append(f'Invalid temperature: {cal_hc.get("temperature")}')
    if cal_hc.get('cal_ece', 1) > 0.15:
        errors.append(f'Calibrated ECE={cal_hc["cal_ece"]:.4f} is high — calibration may have failed')

# Check isotonic regressor
if not os.path.exists('/kaggle/working/isotonic_regressor.pkl'):
    errors.append('isotonic_regressor.pkl is MISSING')

# Check plots
for plot in ['reliability_diagrams.png', 'confidence_bands.png']:
    if not os.path.exists(f'/kaggle/working/{plot}'):
        errors.append(f'Missing plot: {plot}')

# Caveat reminder
print('─' * 55)
print('CALIBRATION CIRCULARITY NOTE:')
print('  Temperature and isotonic were fitted AND evaluated on the SAME')
print('  validation set. Reported post-calibration ECE is slightly optimistic.')
print('  With 100K+ val samples this bias is small but nonzero.')
print('─' * 55)

health = {
    'notebook'     : '06_calibration',
    'status'       : 'PASS' if not errors else 'FAIL',
    'errors'       : errors,
    'raw_ece'      : round(float(raw_ece), 5),
    'cal_ece'      : round(float(best_ece), 5),
    'temperature'  : round(float(T_opt), 4),
    'method'       : CALIBRATION_METHOD,
    'circularity_caveat': 'Fitted and evaluated on same val set',
}

with open('/kaggle/working/health_check_nb06.json', 'w') as f:
    json.dump(health, f, indent=2)

if errors:
    print('❌ HEALTH CHECK FAILED:')
    for e in errors:
        print(f'   • {e}')
else:
    print('✅ HEALTH CHECK PASSED')
    print(f'   Method     : {CALIBRATION_METHOD}')
    print(f'   Temperature: {T_opt:.4f}')
    print(f'   Raw ECE    : {raw_ece:.5f}')
    print(f'   Cal ECE    : {best_ece:.5f}')